In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np

P = "/content/drive/MyDrive/esci/dataset_v2.3/esci_electronics_v2.3_"
queries = pd.read_parquet(P + "queries.parquet")
judg    = pd.read_parquet(P + "judgments.parquet")
corpus  = pd.read_parquet(P + "corpus.parquet")
print("files:", len(queries), len(judg), len(corpus))          # 5665 100740 51318
assert corpus["product_id"].is_unique

# join: judgments + queries (query_id), then + corpus (product_id)
PCOLS = ["product_id", "product_title", "product_brand", "product_color",
         "product_bullet_point", "product_description"]
df = (judg.merge(queries[["query_id", "query", "small"]], on="query_id", how="left")
          .merge(corpus[PCOLS], on="product_id", how="left"))

assert len(df) == len(judg), "join changed the row count"
assert df["query"].notna().all() and df["product_title"].notna().all()

print("joined rows:", len(df))
print("queries per split:", df.groupby("split")["query_id"].nunique().to_dict())   # train 4370, test 1295
print("test small_version queries:", queries.query("split=='test' and small==1").shape[0])   # 507
print("labels:", df["esci_label"].value_counts().to_dict())
print("gain check:", df.groupby("esci_label")["gain"].first().to_dict())    # E 1.0, S 0.1, C 0.01, I 0.0
df[["query", "product_title", "esci_label", "gain"]].head()

Mounted at /content/drive
files: 5665 100740 51318
joined rows: 100740
queries per split: {'test': 1295, 'train': 4370}
test small_version queries: 507
labels: {'E': 67419, 'S': 23141, 'I': 5651, 'C': 4529}
gain check: {'C': 0.01, 'E': 1.0, 'I': 0.0, 'S': 0.1}


,query,product_title,esci_label,gain
0,$150 laptop not previews,"Acer Aspire 5 Slim Laptop, 15.6 inches Full HD...",E,1.0
1,$150 laptop not previews,"HP 2020 15.6 inches HD LED Display, Intel Pent...",E,1.0
2,$150 laptop not previews,"ASUS VivoBook 15 Thin and Light Laptop, 15.6” ...",E,1.0
3,$150 laptop not previews,"HP Stream 14-inch Laptop, Intel Celeron N4000,...",E,1.0
4,$150 laptop not previews,"HP Stream 14-Inch Laptop, Intel Celeron N4000,...",E,1.0


In [2]:
from sklearn.model_selection import train_test_split
import os

SEED = 42
tr_q = queries[queries["split"] == "train"]
fit_ids, val_ids = train_test_split(tr_q["query_id"], test_size=0.15,
                                    random_state=SEED, stratify=tr_q["scope_group"])
fit_ids, val_ids = set(fit_ids), set(val_ids)

df["part"] = np.where(df["split"] == "test", "test",
              np.where(df["query_id"].isin(val_ids), "val", "train"))

# checks
assert not (fit_ids & val_ids), "a query is in both train and val"
print("queries per part:", df.groupby("part")["query_id"].nunique().to_dict())
print("rows per part:", df["part"].value_counts().to_dict())
print("\nlabel mix per part:\n", pd.crosstab(df["part"], df["esci_label"], normalize="index").round(3))

# save to Drive so you never redo this
OUT = "/content/drive/MyDrive/esci/processed"
os.makedirs(OUT, exist_ok=True)
df.to_parquet(f"{OUT}/rerank_pairs_v2.3.parquet")
pd.DataFrame({"query_id": sorted(val_ids)}).to_csv(f"{OUT}/val_query_ids_seed42.csv", index=False)
print("\nsaved:", os.listdir(OUT))

queries per part: {'test': 1295, 'train': 3714, 'val': 656}
rows per part: {'train': 65852, 'test': 23147, 'val': 11741}

label mix per part:
 esci_label      C      E      I      S
part                                  
test        0.049  0.629  0.061  0.261
train       0.042  0.680  0.056  0.222
val         0.050  0.690  0.047  0.214

saved: ['rerank_pairs_v2.3.parquet', 'val_query_ids_seed42.csv']
